# Sarcasm Verification & Cleaning

## Project role

This notebook is **separate from the seven-class emotion pipeline**.

### Sarcasm datasets
- **MUStARD** — multimodal/contextual sarcasm
- **News Headlines Sarcasm Dataset** — text sarcasm

### Explicit boundaries
- Sarcasm is **not** an eighth emotion.
- GoEmotions is **not** used here.
- SAMM is **not** used here.
- The finalized 95,504-row emotion manifest is **not modified** by this notebook.

The output is a binary sarcasm label:
- `0 = non_sarcastic`
- `1 = sarcastic`


In [1]:
import json
import hashlib
import re
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
PROJECT_DIR = Path(r"C:\New folder\New Emodect")

DATASETS_DIR = PROJECT_DIR / "datasets"
CLEANED_DIR = PROJECT_DIR / "cleaned_metadata"
SARCASM_DIR = CLEANED_DIR / "sarcasm"

MUSTARD_DIR = DATASETS_DIR / "MUStARD"
NEWS_SARCASM_DIR = (
    DATASETS_DIR
    / "rmisra"
    / "news-headlines-dataset-for-sarcasm-detection"
)

SARCASM_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("MUStARD:", MUSTARD_DIR)
print("News sarcasm:", NEWS_SARCASM_DIR)
print("Output:", SARCASM_DIR)


Project: C:\New folder\New Emodect
MUStARD: C:\New folder\New Emodect\datasets\MUStARD
News sarcasm: C:\New folder\New Emodect\datasets\rmisra\news-headlines-dataset-for-sarcasm-detection
Output: C:\New folder\New Emodect\cleaned_metadata\sarcasm


In [3]:
SARCASM_LABELS = {
    0: "non_sarcastic",
    1: "sarcastic",
}

print("Sarcasm pipeline: MUStARD + News Headlines")
print("Output labels:", SARCASM_LABELS)
print("Emotion mapping: NONE")


Sarcasm pipeline: MUStARD + News Headlines
Output labels: {0: 'non_sarcastic', 1: 'sarcastic'}
Emotion mapping: NONE


In [4]:
def normalize_text(text):
    if text is None:
        return ""
    text = str(text).replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8", errors="ignore")
    ).hexdigest()


def find_first_value(record, keys):
    if not isinstance(record, dict):
        return None
    for key in keys:
        if key in record and record[key] is not None:
            return record[key]
    return None


def normalize_sarcasm_label(value):
    if value is None:
        return None

    if isinstance(value, bool):
        return int(value)

    if isinstance(value, (int, float)) and value in (0, 1):
        return int(value)

    text = str(value).strip().lower()

    if text in {"1", "true", "yes", "sarcastic", "sarcasm"}:
        return 1

    if text in {
        "0", "false", "no", "non-sarcastic",
        "non_sarcastic", "nonsarcastic", "not sarcastic"
    }:
        return 0

    return None


In [5]:
def discover_json_files(dataset_dir):
    return sorted(Path(dataset_dir).rglob("*.json"))


for name, path in {
    "MUStARD": MUSTARD_DIR,
    "NewsHeadlinesSarcasm": NEWS_SARCASM_DIR,
}.items():
    files = discover_json_files(path)
    print(f"{name}: {len(files)} JSON file(s)")
    for f in files:
        print("  ", f)


MUStARD: 1 JSON file(s)
   C:\New folder\New Emodect\datasets\MUStARD\data\sarcasm_data.json
NewsHeadlinesSarcasm: 2 JSON file(s)
   C:\New folder\New Emodect\datasets\rmisra\news-headlines-dataset-for-sarcasm-detection\versions\2\Sarcasm_Headlines_Dataset.json
   C:\New folder\New Emodect\datasets\rmisra\news-headlines-dataset-for-sarcasm-detection\versions\2\Sarcasm_Headlines_Dataset_v2.json


## MUStARD parser

The parser preserves the multimodal/contextual information available in the source record. It does **not** collapse MUStARD into the emotion taxonomy.


In [6]:
def parse_mustard_json(filepath):
    records = []

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
    except Exception as exc:
        return [], {"status": "JSON_READ_ERROR", "error": str(exc)}

    if isinstance(data, list):
        raw_records = data
    elif isinstance(data, dict):
        if all(isinstance(v, dict) for v in data.values()):
            raw_records = []
            for record_id, value in data.items():
                item = dict(value)
                item["_record_id"] = str(record_id)
                raw_records.append(item)
        else:
            raw_records = [data]
    else:
        return [], {
            "status": "UNSUPPORTED_JSON_STRUCTURE",
            "error": str(type(data)),
        }

    for index, record in enumerate(raw_records):
        if not isinstance(record, dict):
            continue

        record_id = find_first_value(
            record, ["_record_id", "id", "utterance_id", "uid", "clip_id"]
        )
        utterance = find_first_value(
            record, ["utterance", "text", "sentence"]
        )
        context = find_first_value(
            record, ["context", "previous_utterances", "dialogue_context"]
        )
        speaker = find_first_value(
            record, ["speaker", "speaker_id"]
        )
        show = find_first_value(
            record, ["show", "source", "series"]
        )
        video_id = find_first_value(
            record, ["video", "video_id", "youtube_id", "youtube"]
        )
        sarcasm_raw = find_first_value(
            record, ["sarcasm", "is_sarcastic", "label", "target"]
        )

        records.append({
            "dataset": "MUStARD",
            "source_file": str(filepath),
            "record_index": index,
            "record_id": (
                str(record_id)
                if record_id is not None
                else f"record_{index}"
            ),
            "utterance": normalize_text(utterance),
            "context": context,
            "speaker": (
                str(speaker) if speaker is not None else None
            ),
            "show": str(show) if show is not None else None,
            "video_id": (
                str(video_id) if video_id is not None else None
            ),
            "article_link": None,
            "sarcasm_raw": sarcasm_raw,
            "sarcasm": normalize_sarcasm_label(sarcasm_raw),
            "raw_record": json.dumps(
                record, ensure_ascii=False
            ),
        })

    return records, {
        "status": "PROCESSED",
        "records": len(records),
    }


## News Headlines sarcasm parser

This parser supports both JSON Lines and ordinary JSON structures while preserving the original record.


In [7]:
def parse_news_sarcasm_json(filepath):
    records = []

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except Exception as exc:
        return [], {"status": "FILE_READ_ERROR", "error": str(exc)}

    parsed = []
    jsonl_ok = True

    for line in lines:
        line = line.strip()
        if not line:
            continue
        try:
            parsed.append(json.loads(line))
        except Exception:
            jsonl_ok = False
            break

    if not jsonl_ok:
        try:
            with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)

            if isinstance(data, list):
                parsed = data
            elif isinstance(data, dict):
                parsed = [data]
            else:
                return [], {
                    "status": "UNSUPPORTED_JSON_STRUCTURE"
                }
        except Exception as exc:
            return [], {
                "status": "JSON_PARSE_ERROR",
                "error": str(exc)
            }

    for index, record in enumerate(parsed):
        if not isinstance(record, dict):
            continue

        headline = find_first_value(
            record, ["headline", "text", "sentence", "title"]
        )
        sarcasm_raw = find_first_value(
            record, ["is_sarcastic", "sarcasm", "label", "target"]
        )
        article_link = find_first_value(
            record, ["article_link", "link", "url"]
        )

        records.append({
            "dataset": "NewsHeadlinesSarcasm",
            "source_file": str(filepath),
            "record_index": index,
            "record_id": f"headline_{index}",
            "utterance": normalize_text(headline),
            "context": None,
            "speaker": None,
            "show": None,
            "video_id": None,
            "article_link": article_link,
            "sarcasm_raw": sarcasm_raw,
            "sarcasm": normalize_sarcasm_label(sarcasm_raw),
            "raw_record": json.dumps(
                record, ensure_ascii=False
            ),
        })

    return records, {
        "status": "PROCESSED",
        "records": len(records),
    }


In [8]:
mustard_records = []
news_records = []

for filepath in discover_json_files(MUSTARD_DIR):
    records, report = parse_mustard_json(filepath)
    mustard_records.extend(records)
    print("MUStARD:", filepath.name, "→", report)

for filepath in discover_json_files(NEWS_SARCASM_DIR):
    records, report = parse_news_sarcasm_json(filepath)
    news_records.extend(records)
    print("News:", filepath.name, "→", report)

mustard_df = pd.DataFrame(mustard_records)
news_df = pd.DataFrame(news_records)

print()
print("MUStARD records:", len(mustard_df))
print("News sarcasm records:", len(news_df))


MUStARD: sarcasm_data.json → {'status': 'PROCESSED', 'records': 690}
News: Sarcasm_Headlines_Dataset.json → {'status': 'PROCESSED', 'records': 26709}
News: Sarcasm_Headlines_Dataset_v2.json → {'status': 'PROCESSED', 'records': 28619}

MUStARD records: 690
News sarcasm records: 55328


In [9]:
def add_content_hash(df):
    df = df.copy()
    df["normalized_text"] = (
        df["utterance"].fillna("").map(normalize_text)
    )
    df["content_hash"] = df["normalized_text"].map(sha256_text)
    return df


mustard_df = add_content_hash(mustard_df)
news_df = add_content_hash(news_df)

print("Content hashes calculated.")


Content hashes calculated.


## Structural QA

Samples with missing/invalid labels or empty text are excluded from the clean manifest and counted in the QA report.


In [10]:
def validate_sarcasm_dataframe(df, dataset_name):
    issues = []

    if df.empty:
        return [{
            "dataset": dataset_name,
            "row": None,
            "issue": "NO_RECORDS",
        }]

    missing_text = (
        df["utterance"].fillna("").str.strip().eq("")
    )

    for index in df.index[missing_text]:
        issues.append({
            "dataset": dataset_name,
            "row": int(index),
            "issue": "MISSING_TEXT",
        })

    missing_label = df["sarcasm"].isna()

    for index in df.index[missing_label]:
        issues.append({
            "dataset": dataset_name,
            "row": int(index),
            "issue": "MISSING_OR_INVALID_SARCASM_LABEL",
        })

    invalid_label = (
        df["sarcasm"].notna()
        & ~df["sarcasm"].isin([0, 1])
    )

    for index in df.index[invalid_label]:
        issues.append({
            "dataset": dataset_name,
            "row": int(index),
            "issue": "INVALID_SARCASM_LABEL",
        })

    return issues


issues = (
    validate_sarcasm_dataframe(mustard_df, "MUStARD")
    + validate_sarcasm_dataframe(
        news_df, "NewsHeadlinesSarcasm"
    )
)

issues_df = pd.DataFrame(issues)

print("Structural issues:", len(issues_df))
display(issues_df.head(100))


Structural issues: 0


""


## Duplicate and label-conflict analysis

Exact duplicate text is detected within each dataset. Before dropping duplicates, conflicting labels are explicitly reported.


In [11]:
def find_label_conflicts(df):
    grouped = (
        df.groupby("content_hash")["sarcasm"]
        .nunique(dropna=True)
    )
    conflicting_hashes = grouped[grouped > 1].index
    return df[
        df["content_hash"].isin(conflicting_hashes)
    ].copy()


mustard_conflicts = find_label_conflicts(mustard_df)
news_conflicts = find_label_conflicts(news_df)

print(
    "MUStARD conflicting duplicate groups:",
    mustard_conflicts["content_hash"].nunique()
)
print(
    "News conflicting duplicate groups:",
    news_conflicts["content_hash"].nunique()
)

if not mustard_conflicts.empty:
    display(mustard_conflicts)
if not news_conflicts.empty:
    display(news_conflicts)


MUStARD conflicting duplicate groups: 1
News conflicting duplicate groups: 0


,dataset,source_file,record_index,record_id,utterance,context,speaker,show,video_id,article_link,sarcasm_raw,sarcasm,raw_record,normalized_text,content_hash
295,MUStARD,C:\New folder\New Emodect\datasets\MUStARD\dat...,295,2_549,Really?,[Honey the Miami vice sound track?],MONICA,FRIENDS,None,None,False,0,"{""utterance"": ""Really?"", ""speaker"": ""MONICA"", ...",Really?,9eb8cd756595cf24e67f7f24fe58246e94fad2f10e6a95...
657,MUStARD,C:\New folder\New Emodect\datasets\MUStARD\dat...,657,2_209,Really?,"[He doesn't need to be sarcastic. I mean, that...",MEMBER-GIRL,SARCASMOHOLICS,None,None,True,1,"{""utterance"": ""Really?"", ""speaker"": ""MEMBER-GI...",Really?,9eb8cd756595cf24e67f7f24fe58246e94fad2f10e6a95...


In [12]:
def clean_sarcasm_dataframe(df):
    df = df.copy()
    df = df[df["sarcasm"].isin([0,1])].copy()
    df = df[df["normalized_text"].str.len() > 0].copy()
    df = df.sort_values(["dataset","source_file","record_index"]).reset_index(drop=True)

    # Conflicting exact-text duplicates are NEVER silently collapsed.
    conflict_hashes = find_label_conflicts(df)["content_hash"].unique()
    if len(conflict_hashes):
        df = df[~df["content_hash"].isin(conflict_hashes)].copy()

    # Deterministic exact-duplicate removal after conflicts are isolated.
    df = df.drop_duplicates(subset=["content_hash"], keep="first")
    return df.reset_index(drop=True)

mustard_clean = clean_sarcasm_dataframe(mustard_df)
news_clean = clean_sarcasm_dataframe(news_df)

print("MUStARD cleaned:", len(mustard_clean))
print("News cleaned:", len(news_clean))
print("Conflicting duplicate groups excluded from clean manifests:",
      len(set(mustard_conflicts["content_hash"]) | set(news_conflicts["content_hash"])))


MUStARD cleaned: 680
News cleaned: 28503
Conflicting duplicate groups excluded from clean manifests: 1


## Leakage grouping

MUStARD retains speaker identity when available. News headlines do not provide defensible speaker identity, so each headline is treated as its own group rather than inventing a subject identity.


In [13]:
def create_group_id(row):
    if row["dataset"] == "MUStARD":
        if pd.notna(row["speaker"]) and str(row["speaker"]).strip():
            return f"MUStARD_SPEAKER_{row['speaker']}"
        if pd.notna(row["video_id"]) and str(row["video_id"]).strip():
            return f"MUStARD_VIDEO_{row['video_id']}"
        return f"MUStARD_RECORD_{row['record_id']}"
    return f"NEWS_HEADLINE_{row['record_id']}"

mustard_clean["group_id"] = mustard_clean.apply(create_group_id, axis=1)
news_clean["group_id"] = news_clean.apply(create_group_id, axis=1)


## Final manifests


In [14]:
FINAL_COLUMNS = [
    "dataset",
    "record_id",
    "utterance",
    "context",
    "speaker",
    "show",
    "video_id",
    "article_link",
    "sarcasm_raw",
    "sarcasm",
    "group_id",
    "content_hash",
    "source_file",
    "record_index",
    "raw_record",
]

for df in [mustard_clean, news_clean]:
    for column in FINAL_COLUMNS:
        if column not in df.columns:
            df[column] = None

mustard_clean = mustard_clean[FINAL_COLUMNS]
news_clean = news_clean[FINAL_COLUMNS]

mustard_path = SARCASM_DIR / "mustard_sarcasm_clean.csv"
news_path = SARCASM_DIR / "news_headlines_sarcasm_clean.csv"

mustard_clean.to_csv(
    mustard_path, index=False, encoding="utf-8-sig"
)
news_clean.to_csv(
    news_path, index=False, encoding="utf-8-sig"
)

print("Saved:", mustard_path)
print("Saved:", news_path)


Saved: C:\New folder\New Emodect\cleaned_metadata\sarcasm\mustard_sarcasm_clean.csv
Saved: C:\New folder\New Emodect\cleaned_metadata\sarcasm\news_headlines_sarcasm_clean.csv


In [15]:
combined_sarcasm = pd.concat(
    [mustard_clean, news_clean],
    ignore_index=True
)

combined_path = (
    SARCASM_DIR
    / "final_sarcasm_training_manifest.csv"
)

combined_sarcasm.to_csv(
    combined_path,
    index=False,
    encoding="utf-8-sig"
)

print("Combined rows:", len(combined_sarcasm))
print("Saved:", combined_path)


Combined rows: 29183
Saved: C:\New folder\New Emodect\cleaned_metadata\sarcasm\final_sarcasm_training_manifest.csv


In [16]:
def sarcasm_distribution(df):
    out = (
        df["sarcasm"]
        .value_counts()
        .rename(index={
            0: "non_sarcastic",
            1: "sarcastic"
        })
        .rename_axis("sarcasm_label")
        .reset_index(name="count")
    )

    if len(out):
        out["percentage"] = (
            out["count"] / out["count"].sum() * 100
        ).round(2)

    return out


print("MUStARD distribution")
display(sarcasm_distribution(mustard_clean))

print("News Headlines distribution")
display(sarcasm_distribution(news_clean))

print("Combined distribution")
display(sarcasm_distribution(combined_sarcasm))


MUStARD distribution


,sarcasm_label,count,percentage
0,sarcastic,343,50.44
1,non_sarcastic,337,49.56


News Headlines distribution


,sarcasm_label,count,percentage
0,non_sarcastic,14951,52.45
1,sarcastic,13552,47.55


Combined distribution


,sarcasm_label,count,percentage
0,non_sarcastic,15288,52.39
1,sarcastic,13895,47.61


In [17]:
def build_sarcasm_qa(original_df, clean_df, dataset_name):
    return {
        "Dataset": dataset_name,
        "Original_Rows": len(original_df),
        "Clean_Rows": len(clean_df),
        "Removed_Rows": len(original_df) - len(clean_df),
        "Missing_Text": int(
            original_df["utterance"]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        ),
        "Invalid_or_Missing_Labels": int(
            original_df["sarcasm"].isna().sum()
        ),
        "Sarcastic_Clean": int(
            (clean_df["sarcasm"] == 1).sum()
        ),
        "Non_Sarcastic_Clean": int(
            (clean_df["sarcasm"] == 0).sum()
        ),
        "Unique_Groups": int(
            clean_df["group_id"].nunique()
        ),
    }


qa_df = pd.DataFrame([
    build_sarcasm_qa(
        mustard_df,
        mustard_clean,
        "MUStARD"
    ),
    build_sarcasm_qa(
        news_df,
        news_clean,
        "NewsHeadlinesSarcasm"
    ),
])

display(qa_df)

qa_path = SARCASM_DIR / "sarcasm_qa_report.csv"
qa_df.to_csv(qa_path, index=False)


,Dataset,Original_Rows,Clean_Rows,Removed_Rows,Missing_Text,Invalid_or_Missing_Labels,Sarcastic_Clean,Non_Sarcastic_Clean,Unique_Groups
0,MUStARD,690,680,10,0,0,343,337,21
1,NewsHeadlinesSarcasm,55328,28503,26825,0,0,13552,14951,26723


In [18]:
print("=" * 90)
print("FINAL SARCASM DATASET VALIDATION")
print("=" * 90)

assert not mustard_clean.empty, "FINAL GATE FAIL — MUStARD clean manifest is empty."
assert not news_clean.empty, "FINAL GATE FAIL — News sarcasm clean manifest is empty."

for name, df in [("MUStARD", mustard_clean), ("NewsHeadlinesSarcasm", news_clean)]:
    assert df["sarcasm"].isin([0,1]).all(), f"{name}: non-binary label remains."
    assert df["utterance"].fillna("").str.strip().str.len().gt(0).all(), f"{name}: empty text remains."
    assert df["content_hash"].is_unique, f"{name}: exact duplicate remains."
    assert df["group_id"].notna().all(), f"{name}: missing group ID."

assert set(combined_sarcasm["dataset"]) == {"MUStARD","NewsHeadlinesSarcasm"}
assert "emotion" not in combined_sarcasm.columns
assert combined_sarcasm["sarcasm"].isin([0,1]).all()

# Cross-dataset duplicate text is reported, never merged across datasets.
cross_hashes = (
    combined_sarcasm.groupby("content_hash")["dataset"]
    .nunique()
)
cross_dataset_duplicate_groups = int((cross_hashes > 1).sum())

# Final QA is saved again after all final transformations.
final_qa = qa_df.copy()
final_qa["Combined_Clean_Rows"] = len(combined_sarcasm)
final_qa["Cross_Dataset_Duplicate_Text_Groups"] = cross_dataset_duplicate_groups
final_qa.to_csv(qa_path, index=False)

print("✓ All sarcasm labels are binary.")
print("✓ No empty sarcasm text samples.")
print("✓ Exact duplicate text is unique within each dataset.")
print("✓ Conflicting duplicate groups were not silently merged.")
print("✓ Dataset identities preserved.")
print("✓ Emotion labels are absent from sarcasm manifests.")
print(f"✓ Cross-dataset duplicate text groups reported separately: {cross_dataset_duplicate_groups}")


# Additional deterministic final QA
for _name, _df in [("MUStARD", mustard_clean), ("NewsHeadlinesSarcasm", news_clean)]:
    _counts = _df["sarcasm"].value_counts().to_dict()
    assert set(_counts).issubset({0, 1})
    assert sum(_counts.values()) == len(_df)

assert len(combined_sarcasm) == len(mustard_clean) + len(news_clean)
assert combined_sarcasm["record_id"].notna().all()
assert combined_sarcasm["group_id"].astype(str).str.len().gt(0).all()

print("\nFinal sarcasm counts:")
print(f"  MUStARD: sarcastic={(mustard_clean['sarcasm']==1).sum():,}, non-sarcastic={(mustard_clean['sarcasm']==0).sum():,}")
print(f"  NewsHeadlinesSarcasm: sarcastic={(news_clean['sarcasm']==1).sum():,}, non-sarcastic={(news_clean['sarcasm']==0).sum():,}")
print(f"  Combined: {len(combined_sarcasm):,}")

print()
print("FINAL SARCASM GATE: PASS")
print("SARCASM VERIFICATION AND CLEANING COMPLETE")
print()
print("Outputs:")
print(" ", mustard_path)
print(" ", news_path)
print(" ", combined_path)
print(" ", qa_path)


FINAL SARCASM DATASET VALIDATION
✓ All sarcasm labels are binary.
✓ No empty sarcasm text samples.
✓ Exact duplicate text is unique within each dataset.
✓ Conflicting duplicate groups were not silently merged.
✓ Dataset identities preserved.
✓ Emotion labels are absent from sarcasm manifests.
✓ Cross-dataset duplicate text groups reported separately: 0

Final sarcasm counts:
  MUStARD: sarcastic=343, non-sarcastic=337
  NewsHeadlinesSarcasm: sarcastic=13,552, non-sarcastic=14,951
  Combined: 29,183

FINAL SARCASM GATE: PASS
SARCASM VERIFICATION AND CLEANING COMPLETE

Outputs:
  C:\New folder\New Emodect\cleaned_metadata\sarcasm\mustard_sarcasm_clean.csv
  C:\New folder\New Emodect\cleaned_metadata\sarcasm\news_headlines_sarcasm_clean.csv
  C:\New folder\New Emodect\cleaned_metadata\sarcasm\final_sarcasm_training_manifest.csv
  C:\New folder\New Emodect\cleaned_metadata\sarcasm\sarcasm_qa_report.csv
